# ONPE ERM2022 – Scraping Actas por Ubigeo
Notebook para extraer **organización política → votos (y porcentaje)** por **Ubigeo** desde:
`https://resultadoshistorico.onpe.gob.pe/ERM2022/EleccionesMunicipales/RePro`.


In [1]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
# Ruta completa al Excel con los ubigeos
EXCEL_UBIGEOS = "/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Scrapping/ubigeo.xlsx"
# Encabezados esperados en el Excel: ubigeo | reg | prov | dist

# API de ONPE: results/04/{ubigeo}  (04 = municipal distrital)
BASE_API_URL = "https://resultadoshistorico.onpe.gob.pe/v1/ERM2022/results/04/"

# Cabecera para requests (buena práctica)
HEADERS = {
    "User-Agent": "KarlaOnpeScraper/1.0 (karla.vega.consultora@iica.int)"
}

# Archivo de salida
OUTPUT_CSV = "resultados_onpe_erm2022_por_distrito.csv"

# Pausa entre llamadas a la API de ONPE (segundos)
SLEEP_BETWEEN_REQUESTS = 0.3


In [3]:
def limpiar_numero(texto: str):
    """
    Convierte '1,234' / '1 234' / '1.234' en int.
    Devuelve None si no se puede.
    """
    if not texto:
        return None
    t = (
        texto.replace(",", "")
             .replace(".", "")
             .replace(" ", "")
    )
    return int(t) if t.isdigit() else None

In [4]:
# ============================================================
# 1) OBTENER UBIGEOS DESDE EXCEL
# ============================================================

def obtener_ubigeos_desde_excel() -> pd.DataFrame:
    """
    Lee el archivo Excel con encabezados:
    ubigeo | reg | prov | dist

    Devuelve un DataFrame con columnas:
    ubigeo, region, provincia, distrito
    """
    print("Cargando ubigeos desde Excel...")

    df = pd.read_excel(EXCEL_UBIGEOS, dtype=str)

    # Verificar que existan las columnas necesarias
    columnas_necesarias = ["ubigeo", "reg", "prov", "dist"]
    for col in columnas_necesarias:
        if col not in df.columns:
            raise ValueError(f"Falta la columna '{col}' en el Excel de ubigeos")

    # Renombrar a los nombres usados en el script
    df = df.rename(columns={
        "reg": "region",
        "prov": "provincia",
        "dist": "distrito",
    })

    # Limpiar ubigeo: quitar espacios y dejar solo dígitos
    df["ubigeo"] = df["ubigeo"].astype(str).str.strip()
    df["ubigeo"] = df["ubigeo"].str.replace(r"\D", "", regex=True)

    # Quitar filas sin ubigeo, quitar duplicados y ordenar
    df = (
        df.dropna(subset=["ubigeo"])
          .drop_duplicates(subset=["ubigeo"])
          .sort_values("ubigeo")
          .reset_index(drop=True)
    )

    print(f"Se cargaron {len(df)} ubigeos desde el Excel.")
    return df


In [5]:
# ============================================================
# 2) LLAMAR A LA API DE ONPE PARA UN UBIGEO
# ============================================================

def obtener_resultados_ubigeo(ubigeo: str):
    """
    Llama a la API de la ONPE:
    https://resultadoshistorico.onpe.gob.pe/v1/ERM2022/results/04/{ubigeo}

    Devuelve una lista de dicts:
    - ubigeo
    - organizacion_politica
    - total_votos
    """
    url = BASE_API_URL + ubigeo
    resp = requests.get(url, headers=HEADERS, timeout=60)
    resp.raise_for_status()
    data = resp.json()

    resultados = []

    for item in data.get("results", []):
        agrup = (item.get("AGRUPACION") or "").strip()
        total_txt = (item.get("TOTAL_VOTOS") or "").strip()

        if not agrup:
            continue

        up = agrup.upper()
        # Excluir totales / blancos / nulos
        if any(x in up for x in [
            "TOTAL DE VOTOS VÁLIDOS",
            "TOTAL DE VOTOS EMITIDOS",
            "VOTOS EN BLANCO",
            "VOTOS NULOS",
        ]):
            continue

        total = limpiar_numero(total_txt)
        if total is None:
            continue

        resultados.append({
            "ubigeo": ubigeo,
            "organizacion_politica": agrup,
            "total_votos": total,
        })

    return resultados

In [6]:
# ============================================================
# 3) MAIN
# ============================================================

def main():
    # 1) Cargar ubigeos distritales desde tu Excel
    df_ubi = obtener_ubigeos_desde_excel()

    # --- OPCIONAL: para probar con pocos distritos al inicio ---
    # df_ubi = df_ubi.head(10)
    # -----------------------------------------------------------

    registros = []

    for _, row in df_ubi.iterrows():
        ubigeo = row["ubigeo"]
        region = row["region"]
        provincia = row["provincia"]
        distrito = row["distrito"]

        print(f"\nUbigeo {ubigeo} - {region} / {provincia} / {distrito}")

        try:
            filas = obtener_resultados_ubigeo(ubigeo)
        except Exception as e:
            print(f"  [ERROR] al consultar ONPE para {ubigeo}: {e}")
            continue

        for r in filas:
            registros.append({
                "ubigeo": ubigeo,
                "region": region,
                "provincia": provincia,
                "distrito": distrito,
                "organizacion_politica": r["organizacion_politica"],
                "total_votos": r["total_votos"],
            })

        time.sleep(SLEEP_BETWEEN_REQUESTS)

    df_final = pd.DataFrame(registros)
    df_final.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\nListo. Se guardó el archivo: {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

Cargando ubigeos desde Excel...
Se cargaron 1838 ubigeos desde el Excel.

Ubigeo 010101 - Amazonas / Chachapoyas / Chachapoyas
  [ERROR] al consultar ONPE para 010101: 500 Server Error: Internal Server Error for url: https://resultadoshistorico.onpe.gob.pe/v1/ERM2022/results/04/010101

Ubigeo 010102 - Amazonas / Chachapoyas / Asuncion

Ubigeo 010103 - Amazonas / Chachapoyas / Balsas

Ubigeo 010104 - Amazonas / Chachapoyas / Cheto

Ubigeo 010105 - Amazonas / Chachapoyas / Chiliquin

Ubigeo 010106 - Amazonas / Chachapoyas / Chuquibamba

Ubigeo 010107 - Amazonas / Chachapoyas / Granada

Ubigeo 010108 - Amazonas / Chachapoyas / Huancas

Ubigeo 010109 - Amazonas / Chachapoyas / La Jalca

Ubigeo 010110 - Amazonas / Chachapoyas / Leimebamba

Ubigeo 010111 - Amazonas / Chachapoyas / Levanto

Ubigeo 010112 - Amazonas / Chachapoyas / Magdalena

Ubigeo 010113 - Amazonas / Chachapoyas / Mariscal Castilla

Ubigeo 010114 - Amazonas / Chachapoyas / Molinopampa

Ubigeo 010115 - Amazonas / Chachapoyas 